In [69]:
import pandas as pd
import numpy as np
import tokenize, ast
import openai
from io import BytesIO
from openai import ChatCompletion, Completion
from fastcore.utils import nested_idx
from tiktoken import encoding_for_model

In [2]:
# %pip install openai==0.28

In [70]:
enc = encoding_for_model("text-davinci-003")
toks = enc.encode("Wait! That's dangerous!")
toks

[21321, 0, 1320, 338, 4923, 0]

In [72]:
[enc.decode_single_token_bytes(o).decode('utf-8') for o in toks]

['Wait', '!', ' That', "'s", ' dangerous', '!']

In [21]:
risk_preamble = """Studying small incidents in coal mines is important because it can lead to important safety improvements and thus reduce the risk for workers. However, not all incidents have the same importance: some are minor events and some are near misses.

Minor events are incidents where at worst minor first aid is required, but the event could not have been worse. 			

Near misses are minor events where at worst minor first aid is required, but if not for some good luck, the incident could have been worse.
 
The element that distinguishes a near miss, is the presence of a much worse consequence that was prevented only by good luck.
 
For example, read this incident: 

AN EMPLOYEE WAS PULLING AN INVOICE FROM AN ENVELOPE, WHILE PULLING HIS POINTER FINGER MADE CONTACT WITH THE BANDING MATERIAL AND SLICED A KNUCKLE. 

This was a paper cut, a minor event. Good luck did not prevent a worse event from happening. 

Alternatively, read this other incident: 
 THE EMPLOYEES LEFT INDEX FINGER WAS PINCHED BETWEEN A PALLET OF MATERIAL AND A LIVE ROLLER AS THE PALLET WAS BEING MOVED FROM THE HOODER SYSTEM. THE TIP OF THE LEFT INDEX FINGER AND PART OF THE FINGERNAIL WAS LACERATED DURING INCIDENT. 

This is a near miss, since it seems that with moving material and parts and someone's hand getting caught, it was good luck that only the tip of the finger and fingernail were injured, so we think that this is an example of a near miss.

We don't expect you to be experts in coal mining incidents or understand all the technical jargon (we don't know what a hooder system is either).

Here are a couple more examples:
WHILE DESCENDING A STEP LADDER, THE EMPLOYEE DRAGGED HAND ACROSS TOP OF LOCKER & CUT HAND. THIS CUT REQUIRED STITCHES.

We think this is a minor event and not a near miss. This was a minor cut.

WHILE DUMPING LOAD TO CONSTRUCT BARM, THE TRUCK LOST POWER AND BACKED OVER A 10' LEDGE. THE OPERATOR STRUCK HIS LEFT KNEE AGAINST THE DASHBOARD RESULTING IN A LACERATION.

We think this is a near miss. The truck went over a 10 foot ledge. That is pretty high. The operator was lucky that he only hit and cut his knee on the dashboard.

ELECTRICIAN WAS LEAVING ELECTRICAL SHOP AND WIND GUSTS OF 40+ MPG SHUT THE DOOR ON HIS FINGER, FRACTURING THE TIP OF HIS FINGER.

We think this is a minor event and not a near miss. The tip of the finger got caught in a door

ELECTRICAL SHOCK - WORKING ON 60 HP MOTOR WHICH WAS LOCKED & TAGGED OUT PER SAFETY REGULATIONS. SAW A FLASH WHICH MAY HAVE BEEN CAUSED BY LIGHTNING - FELT SHOCK - HE WAS SENT TO E.R. - EVALUATED & RELEASED - RETURNED TO WORK WITH NO RESTRICTIONS.

We think this is a near miss. Electrical shocks are serious issues and the employee was lucky this was not a worse event. 

We hope the task makes sense, and we appreciate your help. This work can be valuable for improving safety in dangerous industries. Please do the best that you can with the coal mining terminology since these were taken from real safety reports. Also, note "EE" and "IE" refer to the employee/injured employee. Thank you again for your help.

 
"""

In [75]:
risk_narrative = """
WHILE HANDLING CURTAIN LINE A NAIL CUT EMPLOYEE LEFT HAND.
"""

In [30]:
c = ChatCompletion.create(
    api_key=API_KEY_CONST,
    model="gpt-3.5-turbo",
    messages=[{"role": "system", "content": risk_preamble},
              {"role": "user", "content": risk_narrative}])

In [31]:
c['choices'][0]['message']['content']

'This incident appears to be a minor event. The employee cut their hand on a nail while handling the curtain line. This injury likely required minor first aid and did not involve a worse consequence that was only prevented by good luck.'

In [26]:
def response(compl): print(nested_idx(compl, 'choices', 0, 'message', 'content'))

In [33]:
response(c)

This incident appears to be a minor event. The employee cut their hand on a nail while handling the curtain line. This injury likely required minor first aid and did not involve a worse consequence that was only prevented by good luck.


In [34]:
print(c.usage)

{
  "prompt_tokens": 776,
  "completion_tokens": 45,
  "total_tokens": 821
}


In [24]:
c = ChatCompletion.create(
    model="gpt-3.5-turbo",
    api_key=API_KEY_CONST,
    messages=[{"role": "system", "content": risk_preamble},
              {"role": "user", "content": risk_narrative},
              {"role": "assistant", 
               "content": "Please respond without echoing the narrative text. \
               Only your judgement on the risk level."}])

# We can add tighter response restrictions into this call via the "assistant"

In [27]:
response(c)

This incident is a minor event.


These responses were all obtained from running an identical API call:

 - This incident seems to be a minor event.

 - This incident seems to be a minor event as it resulted in a simple nail cut to the employee's left hand.

 - This is a minor event.
 
 - This incident is considered a minor event.
 
 - This incident appears to be a minor event as it resulted in a cut to the employee's hand without any indication of a more severe outcome being narrowly avoided.
 
 - This incident seems to be a minor event as it only resulted in a cut to the employee's hand. There doesn't appear to be a potentially worse consequence that was narrowly avoided by luck.

In [28]:
print(c.usage)

{
  "prompt_tokens": 797,
  "completion_tokens": 7,
  "total_tokens": 804
}


In [15]:
narratives = []
# ALL VOTES LISTED FOR *FOR* A NEAR-MISS
# Voted 0/9 
narratives.append("""
EMPLOYEE STATES THAT EE WAS CLEANING THE #5 SCRUBBER BEFORE EE WENT ON BREAK.  AT BREAK TIME, EE TOOK GLOVE OFF AND NOTICED BLOOD INSIDE EE'S GLOVE FROM A LACERATION TO RIGHT MIDDLE FINGER. EE IS UNSURE HOW IT OCCURRED. 
""")

# Voted 2/8
narratives.append("""
EMPLOYEE WAS LOADING A PALLET INTO THE BEMIS PALLETIZER WHEN HE DROPPED THE PALLET AND PINCHED HIS FINGER AGAINST A METAL STRIP CAUSING A LACERATION TO THE 5TH FINGER ON HIS LEFT HAND. EE RECEIVED 7 STITCHES AND RETURNED TO WORK WITH NO RESTRICTIONS.
""")

# Voted 4/8
narratives.append("""
EMPLOYEE CUT HIS LEFT ARM BETWEEN THE WRIST AND ELBOW. THE INJURY OCCURRED WHILE HE WAS CUTTING A 1 INCH WATER HOSE WITH A RAZOR KNIFE AT THE REAR OF THE # 2 LIME KILN STAIRWAY. HE RECEIVED 3 STITCHES AND RETURNED TO WORK.
""")

# Voted 6/9
narratives.append("""
WHILE USING A 3/8" ROD TO HELP CLEAR CHUNKS OF MATERIAL FROM A 10" PORTHOLE CEMENT UNEXPECTEDLY FOLLOWED AND IT IS BELIEVED THAT IS WHEN CEMENT/CEMENT DUST GOT INTO HIS EYE.  HIS EYE BECAME IRRITATED AS THE DAY WENT ON AND AT 1320HRS HE WAS TRANSPORTED TO THE EYE DOCTOR AND DIAGNOSED WITH A SCRATCH TO HIS LEFT EYE AND EYE DROPS WERE PRESCRIBED.
""")

# Voted 7/7
narratives.append("""
EMPLOYEE REMOVED PLASTIC TRIM TO TRY AND FIX AN INTERMITTENTLY WORKING WINDSHIELD WIPER ON A DOZER. IN REMOVING THE PLASTIC, A SECTION BROKE OFF. EMPLOYEE WAS GOING TO SET THE PIECE OUTSIDE OF THE DOZER, AND WHILE EXITING THE MACHINERY EE HIT THE DASHBOARD WITH THE TRIM AND THE SHARP EDGE PENETRATED THROUGH EE'S SHIRT AND STUCK EE IN EE'S RIGHT ARM JUST BELOW EE'S BICEP.
""")

print(len(narratives))

5


In [80]:
N = len(narratives)
comps = []

for i in range(0, N):
    c = ChatCompletion.create(
        model="gpt-3.5-turbo",
        api_key=API_KEY_CONST,
        messages=[{"role": "system", "content": risk_preamble},
                  {"role": "user", "content": narratives[i]},
                  {"role": "assistant", 
                   "content": "Please respond with only and exclusively one word: 'True' or 'False'. \
                   'True' if and only if it is a near-miss event."}])
    comps.append(c)



In [81]:
for c in comps:
    response(c)

False
False
False
True
False


In [82]:
for c in comps:
    print(c.usage)

{
  "prompt_tokens": 865,
  "completion_tokens": 1,
  "total_tokens": 866
}
{
  "prompt_tokens": 873,
  "completion_tokens": 1,
  "total_tokens": 874
}
{
  "prompt_tokens": 869,
  "completion_tokens": 1,
  "total_tokens": 870
}
{
  "prompt_tokens": 912,
  "completion_tokens": 1,
  "total_tokens": 913
}
{
  "prompt_tokens": 913,
  "completion_tokens": 1,
  "total_tokens": 914
}


In [57]:
def get_comps(narratives):

    N = len(narratives)
    comps = []

    for i in range(0, N):
        c = ChatCompletion.create(
            model="gpt-3.5-turbo",
            api_key=API_KEY_CONST,
            messages=[{"role": "system", "content": risk_preamble},
                      {"role": "user", "content": narratives[i]},
                      {"role": "assistant", 
                       "content": "Please respond with only and exclusively one word: 'True' or 'False'. \
                       'True' if and only if it is a near-miss event."}])
        comps.append(c)
    return comps

In [83]:
X = 10
comps_matrix = []

for i in range(0, X):
    comps_matrix.append(get_comps(narratives))
print(len(comps_matrix))
print(len(comps_matrix[0]))

10
5


In [67]:
over_time = []
# def response(compl): print(nested_idx(compl, 'choices', 0, 'message', 'content'))
print([nested_idx(c, 'choices', 0, 'message', 'content') for c in comps])

['False', 'False', 'False', 'True.', 'True']
['True', 'False', 'False', 'True', 'True']
['False', 'False', 'False', 'False', 'True']
['False', 'False', 'False', 'True', 'False']
['False', 'False', 'False', 'False', 'True']
['False', 'False', 'True', 'True', 'True']
['True', 'True', 'True', 'True', 'True']
['False', 'False', 'False', 'False', 'True']
['False', 'True', 'False', 'False', 'True']
['False', 'False', 'False', 'True', 'True']


In [16]:
def call_api(prompt, model="gpt-3.5-turbo"):
    msgs = [{"role": "user", "content": prompt}]
    try: return ChatCompletion.create(model=model, messages=msgs)
    except openai.error.RateLimitError as e:
        retry_after = int(e.headers.get("retry-after", 60))
        print(f"Rate limit exceeded, waiting for {retry_after} seconds...")
        time.sleep(retry_after)
        return call_api(params, model=model)

In [39]:

call_api("What's the world's funniest joke? Has there ever been any scientific analysis?")

<OpenAIObject chat.completion id=chatcmpl-8xSiiKTvPNJWlLHFUw1xZ6A2vZCQA at 0x7f6070219400> JSON: {
  "id": "chatcmpl-8xSiiKTvPNJWlLHFUw1xZ6A2vZCQA",
  "object": "chat.completion",
  "created": 1709182444,
  "model": "gpt-3.5-turbo-0125",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "There have been several attempts to find the world's funniest joke through scientific analysis. One such study was conducted in 2002 by psychologist Richard Wiseman, who created a website called LaughLab to gather and analyze jokes from around the world.\n\nAfter receiving over 40,000 jokes and 1.5 million ratings, Wiseman and his team concluded that the world's funniest joke is as follows:\n\n\"Two hunters are out in the woods when one of them collapses. He doesn't seem to be breathing and his eyes are glazed. The other guy whips out his phone and calls the emergency services. He gasps, \"My friend is dead! What can I do?\" The operator says, \"C

In [25]:
c = Completion.create(prompt="Australian Jeremy Howard is ",
                      model="gpt-3.5-turbo-instruct", echo=True)
#                       model="gpt-3.5-turbo-instruct", echo=True, logprobs=5)

## PyTorch and Huggingface

### Your GPU options

Free:

- Kaggle (2 GPUs, low RAM)
- Colab

Buy:

- Buy 1-2 NVIDIA 24GB GPUs
    - GTX 3090 used (USD700-USD800), or 4090 new (USD2000)
- Alternatively buy one NVIDIA A6000 with 48GB RAM (but this mightn't be faster than 3090/4090)
- Mac with lots of RAM (much slower than NVIDIA; M2 Ultra is best)

In [96]:
from transformers import AutoModelForCausalLM,AutoTokenizer
import torch

- [HF leaderboard](https://huggingface.co/spaces/HuggingFaceH4/open_llm_leaderboard)
- [fasteval](https://fasteval.github.io/FastEval/)

In [11]:
mn = "meta-llama/Llama-2-7b-hf"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(mn, device_map=0, load_in_8bit=True)

In [4]:
tokr = AutoTokenizer.from_pretrained(mn)
prompt = "Jeremy Howard is a "
toks = tokr(prompt, return_tensors="pt")

In [5]:
toks

{'input_ids': tensor([[    1,  5677,  6764, 17430,   338,   263, 29871]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}

In [6]:
tokr.batch_decode(toks['input_ids'])

['<s> Jeremy Howard is a ']

In [27]:
%%time
res = model.generate(**toks.to("cuda"), max_new_tokens=15).to('cpu')
res

CPU times: user 1.34 s, sys: 0 ns, total: 1.34 s
Wall time: 1.34 s


tensor([[    1,  5677,  6764, 17430,   338,   263, 29871, 29941, 29900,  1629,
          2030,  9870, 15640,   322,  4823, 13236, 29889,   940,   756,  1063,
         15859,  6351]])

In [13]:
tokr.batch_decode(res)

['<s> Jeremy Howard is a 28-year-old Australian AI researcher and entrepreneur']

In [28]:
model = AutoModelForCausalLM.from_pretrained(mn, device_map=0, torch_dtype=torch.bfloat16)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [29]:
%%time
res = model.generate(**toks.to("cuda"), max_new_tokens=15).to('cpu')
res

CPU times: user 390 ms, sys: 431 µs, total: 391 ms
Wall time: 389 ms


tensor([[    1,  5677,  6764, 17430,   338,   263, 29871, 29896, 29945, 29899,
          6360, 29899,  1025,   515,   278, 10261,  1058,   338,   263,  1583,
         29899, 29873]])

In [7]:
model = AutoModelForCausalLM.from_pretrained('TheBloke/Llama-2-7b-Chat-GPTQ', device_map=0, torch_dtype=torch.float16)

In [9]:
%%time
res = model.generate(**toks.to("cuda"), max_new_tokens=15).to('cpu')
res

CPU times: user 270 ms, sys: 0 ns, total: 270 ms
Wall time: 269 ms


tensor([[    1,  5677,  6764, 17430,   338,   263, 29871, 29941, 29945, 29899,
          6360, 29899,  1025,   767,   515,   278,  3303,  3900,  1058,   471,
         24383,   297]])

In [42]:
mn = 'TheBloke/Llama-2-13B-GPTQ'
model = AutoModelForCausalLM.from_pretrained(mn, device_map=0, torch_dtype=torch.float16)

In [43]:
%%time
res = model.generate(**toks.to("cuda"), max_new_tokens=15).to('cpu')
res

CPU times: user 341 ms, sys: 8.2 ms, total: 349 ms
Wall time: 348 ms


tensor([[    1,  5677,  6764, 17430,   338,   263, 29871, 29906, 29900, 29896,
         29947, 29899, 29906, 29900, 29896, 29929, 23004,  1182,   523,  1102,
         10170,   322]])

In [44]:
def gen(p, maxlen=15, sample=True):
    toks = tokr(p, return_tensors="pt")
    res = model.generate(**toks.to("cuda"), max_new_tokens=maxlen, do_sample=sample).to('cpu')
    return tokr.batch_decode(res)

In [51]:
gen(prompt, 50)

['<s> Jeremy Howard is a 16-year veteran of Silicon Valley, and a co-founder of Kaggle, a market place for predictive modeling.\nHis company, kaggle.com, has become to data science competitions what']

[StableBeluga-7B](https://huggingface.co/stabilityai/StableBeluga-7B)

In [31]:
mn = "stabilityai/StableBeluga-7B"
model = AutoModelForCausalLM.from_pretrained(mn, device_map=0, torch_dtype=torch.bfloat16)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [32]:
sb_sys = "### System:\nYou are Stable Beluga, an AI that follows instructions extremely well. Help as much as you can.\n\n"

In [33]:
def mk_prompt(user, syst=sb_sys): return f"{syst}### User: {user}\n\n### Assistant:\n"

In [34]:
ques = "Who is Jeremy Howard?"

In [35]:
gen(mk_prompt(ques), 150)

['<s> ### System:\nYou are Stable Beluga, an AI that follows instructions extremely well. Help as much as you can.\n\n### User: Who is Jeremy Howard?\n\n### Assistant:\n Jeremy Howard is an Australian entrepreneur, computer scientist, and co-founder of the Machine Learning and Deep Learning startup company, Fast.ai. He is also known for his work in open source software and has co-led the development of several widely used libraries for deep learning and machine learning.</s>']

[OpenOrca/Platypus 2](https://huggingface.co/Open-Orca/OpenOrca-Platypus2-13B)

In [28]:
mn = 'TheBloke/OpenOrca-Platypus2-13B-GPTQ'
model = AutoModelForCausalLM.from_pretrained(mn, device_map=0, torch_dtype=torch.float16)

In [27]:
def mk_oo_prompt(user): return f"### Instruction: {user}\n\n### Response:\n"

In [30]:
gen(mk_oo_prompt(ques), 150)

['<s> ### Instruction: Who is Jeremy Howard?\n\n### Response:\n\nJeremy Howard is a notable British computer scientist, entrepreneur, and former professional poker player. He is best known for co-founding several successful companies in the fields of data science, artificial intelligence, and machine learning. \n\nOne of his most well-known ventures is the data science platform, fast.ai, which he co-founded in 2017. Additionally, he co-founded the machine learning company, Kaggle, in 2011, which was acquired by Google in 2017. Howard is a renowned figure in the data science and AI community, having contributed significantly to the research and development of these technologies and being an']

### Retrieval augmented generation

In [53]:
from wikipediaapi import Wikipedia

In [54]:
wiki = Wikipedia('JeremyHowardBot/0.0', 'en')
jh_page = wiki.page('Jeremy_Howard_(entrepreneur)').text
jh_page = jh_page.split('\nReferences\n')[0]

In [59]:
print(jh_page[:500])

Jeremy Howard (born 13 November 1973) is an Australian data scientist, entrepreneur, and educator.He is the co-founder of fast.ai, where he teaches introductory courses, develops software, and conducts research in the area of deep learning.
Previously he founded and led Fastmail, Optimal Decisions Group, and Enlitic. He was President and Chief Scientist of Kaggle.
Early in the COVID-19 epidemic he was a leading advocate for masking.

Early life
Howard was born in London, United Kingdom, and move


In [60]:
len(jh_page.split())

613

In [58]:
ques_ctx = f"""Answer the question with the help of the provided context.

## Context

{jh_page}

## Question

{ques}"""

In [60]:
res = gen(mk_prompt(ques_ctx), 300)

In [62]:
print(res[0].split('### Assistant:\n')[1])

 Jeremy Howard is an Australian data scientist, entrepreneur, and educator known for his work in deep learning. He is the co-founder of fast.ai, where he teaches courses, develops software, and conducts research in the field. Before co-founding fast.ai, he was the President and Chief Scientist of Kaggle, the CEO of Fastmail and Optimal Decisions Group, and has a background in management consulting.</s>


In [64]:
from sentence_transformers import SentenceTransformer

In [67]:
emb_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=0)

In [93]:
jh = jh_page.split('\n\n')[0]
print(jh)

Jeremy Howard (born 13 November 1973) is an Australian data scientist, entrepreneur, and educator.He is the co-founder of fast.ai, where he teaches introductory courses, develops software, and conducts research in the area of deep learning.
Previously he founded and led Fastmail, Optimal Decisions Group, and Enlitic. He was President and Chief Scientist of Kaggle.
Early in the COVID-19 epidemic he was a leading advocate for masking.


In [74]:
tb_page = wiki.page('Tony_Blair').text.split('\nReferences\n')[0]

In [98]:
tb = tb_page.split('\n\n')[0]
print(tb[:380])

Sir Anthony Charles Lynton Blair  (born 6 May 1953) is a British politician who served as Prime Minister of the United Kingdom from 1997 to 2007 and Leader of the Labour Party from 1994 to 2007. He served as Leader of the Opposition from 1994 to 1997 and had various shadow cabinet posts from 1987 to 1994. Blair was Member of Parliament (MP) for Sedgefield from 1983 to 2007. He 


In [128]:
q_emb,jh_emb,tb_emb = emb_model.encode([ques,jh,tb], convert_to_tensor=True)

In [129]:
tb_emb.shape

torch.Size([384])